# SoilGrids catalog explorer

Before fetching, it helps to know **what** SoilGrids publishes: which
properties, at which depths, and in what units. The `soilgrids` catalog is
config-as-code — a pydantic `Catalog` you can query offline (no network) to
build a request and to read the scaled-integer unit metadata.

This notebook lists the curated properties, renders their unit / depth /
quantile metadata as tables, and shows the did-you-mean error you get for a
typo.

## Load the catalog

`Catalog()` loads the bundled property catalog from disk. `parameters()` returns
the curated property ids.

In [ ]:
import pandas as pd

from earthlens.soilgrids import Catalog

catalog = Catalog()
catalog.parameters()

## Properties, units and scale factors

Each property row records its WCS endpoint, published depths and quantiles, and
the scaled-integer unit metadata (`unit`, `mapped_units`, `scale_factor`). The
table below is what you need to convert a downloaded raster to a physical unit:
divide the stored value by `scale_factor` to turn `mapped_units` into `unit`.

In [ ]:
rows = [
    {
        "property": p,
        "title": catalog.get(p).title,
        "stored (mapped_units)": catalog.get(p).mapped_units,
        "scale_factor": catalog.get(p).scale_factor,
        "unit": catalog.get(p).unit,
    }
    for p in catalog.parameters()
]
pd.DataFrame(rows).set_index("property")

## Depths and quantiles

Ten of the eleven properties publish the six **standard depth intervals**; the
`ocs` (organic carbon stock) property is the exception — a single `0-30cm`
interval. Every property offers the same five statistical **layers**.

In [ ]:
depth_rows = [
    {
        "property": p,
        "n_depths": len(catalog.get(p).depths),
        "depths": ", ".join(catalog.get(p).depths),
    }
    for p in catalog.parameters()
]
pd.DataFrame(depth_rows).set_index("property")

In [ ]:
# The quantile / layer tokens are the same for every property.
catalog.get("phh2o").quantiles

## Did-you-mean on a typo

`get()` raises a `ValueError` with a suggestion when a property id is not
curated, so a typo fails fast with a hint rather than a silent miss.

In [ ]:
try:
    catalog.get("clayy")
except ValueError as err:
    print(err)

## Takeaway

- `Catalog().parameters()` lists the 11 curated properties; `Catalog().get(id)`
  returns a row with its depths, quantiles, and unit / scale metadata.
- Divide a downloaded raster by `scale_factor` to convert to the physical
  `unit`.
- Feed the property / depth / quantile ids straight into
  `EarthLens(data_source="soilgrids", variables=, depths=, quantiles=)`.